A notebook is a useful place to develop an idea: prose, code, and the results of running that code can live together. But a notebook is also a JSON document, and a small text edit can be awkward when the only interface available is a terminal or an agent working through files.

Atlas treats the notebook as the source of a post, not as an intermediate file to be regenerated. The notebook CLI gives me a way to find and edit its cells, inspect saved results, and run code deliberately without hand-editing notebook JSON. This is the second article in [Building Atlas](../collections/building-atlas.qmd), following [the content CMS](organizing-notebooks-and-metadata.qmd).

## Separate the article from its record

The CMS tracks a post's stable ID, title, description, publication state, and other listing fields. The notebook under `content/posts/` holds its Markdown and code cells, outputs, attachments, and cell options. I can edit that notebook in Jupyter or VS Code, or operate on it with `at notebook`.

These are ordinary notebooks: there is no required front matter or special planning cell. During sync, the CMS generates a frontend copy with document metadata, and Quarto builds the page from that copy. Neither editing a cell nor building the site runs its code. That distinction matters when a post contains saved results.

All the commands below run from the repository root. The examples use `post/understanding-attention`; substitute an existing post ID returned by `map` or the CMS. Notebook commands also accept a unique title or source stem, but exact IDs make scripts less ambiguous.

## Find the right cell

I start by surveying the available notebooks and searching their source:

```sh
uv run at notebook map
uv run at notebook find "attention weights"
uv run at notebook outline post/understanding-attention
uv run at notebook count post/understanding-attention
```

`find` searches cell source, not stored outputs. `outline` returns the notebook revision and Markdown headings with their levels, zero-based cell indices, and one-based line numbers within each cell. It skips headings in code cells, code blocks, HTML blocks, and comments. That gives me an address for an edit without dumping a whole notebook.

```sh
uv run at notebook cat post/understanding-attention --index 0 --context 1 --with-revision
uv run at notebook cat post/understanding-attention --index 0:4
uv run at notebook cat post/understanding-attention --label fig-attention
```

`cat` also accepts tags and negative indices. Source is truncated at 4,096 characters per cell by default; `--offset` and `--limit` page through it, and `--limit 0` returns the full source. Use `--with-outputs` when the saved results matter. An outline is a snapshot: after inserting or removing a cell, I refresh it before using an index again.

## Edit a cell without replacing the notebook

Once I have read the target and its revision, I can replace just that cell's source. For a multiline edit, a quoted heredoc supplies the content without opening an editor:

```sh
uv run at notebook edit-cell post/understanding-attention --index 0 \
  --expected-revision 'notebook:<revision-from-cat>' --file - <<'CELL'
## Attention

An attention layer combines information from the input.
CELL
```

The token in this example is a placeholder: use the `notebook:` token returned by the latest `cat --with-revision` or `outline`. It is distinct from the CMS record revision returned by `cms post show`. If another save changes the notebook first, the expected revision rejects the stale write. I re-read and reconcile the change instead of dropping the check. Failed editor input is kept under `.atlas/cms/edits/`.

`edit-cell` replaces the entire selected source, so the replacement must include any existing Quarto options, labels, and prose I want to keep. It preserves the cell's metadata, attachments, and saved outputs. That also means an edited code cell can retain an old result until I execute it or clear it.

For structural changes, there are explicit operations:

```sh
uv run at notebook insert-cell post/understanding-attention --before 2 --content "## Results"
uv run at notebook append-cell post/understanding-attention --type code --file /path/to/example.py
uv run at notebook tag post/understanding-attention --index 1 --add hide-input
uv run at notebook remove-cell post/understanding-attention --index 3
```

When acting on an earlier read, pass the current `--expected-revision` to each mutation and refresh the token after a successful save. A unique label or tag can be useful when an index is likely to move. Inspect tag matches before removal: `remove-cell --tag` removes every matching cell. Cell input is limited to 20,000 characters, so a longer article can be split across Markdown cells.

## Bring figures into managed storage

A figure can be imported from a local file and inserted as a Markdown cell:

```sh
uv run at notebook figure post/understanding-attention /path/to/plot.svg \
  --caption "Attention weights" --alt "A heatmap of attention weights."
```

Atlas copies the image into `content/images/` and leaves the original in place. It appends the figure by default; `--index` inserts it at a chosen position. The command returns a `fig-...` label, which can identify the figure later:

```sh
uv run at notebook figure post/understanding-attention /path/to/new-plot.png \
  --replace --label fig-attention
```

Use the actual returned label. Replacement changes the image link while preserving the caption, alt text, sizing, label, and surrounding prose. PNG, JPEG, SVG, WebP, and GIF are supported for notebook figures. Keep older managed images if other content still references them.

## How a cell edit becomes a safe save

The CLI in `src/atlas/notebook_cli.py` handles arguments and input; the operations in `src/atlas/notebook_ops.py` work with the notebook; `Store` commits the result. For `edit-cell`, the path through those layers is concrete:

1. Under the store lock, `load()` resolves the post, reads the record and notebook bytes, and checks any supplied notebook revision. The token is `notebook:` followed by the SHA-256 digest of those exact bytes.
2. `unique_index()` selects the cell from the loaded notebook. The CLI obtains the replacement source, then calls `update()` with the loaded token. This matters when input comes from an editor: the file may change while the editor is open.
3. `update()` loads again under the lock, checks that token, deep-copies the notebook, and applies a transform. `set_source()` changes the chosen cell's `source` field and enforces the 20,000-character limit; other cell fields remain intact.
4. `save()` serializes the modified notebook and calls `Store.write_notebook()` with both the loaded CMS record revision and the digest of the original notebook bytes. The store reacquires the writer lock and checks both before committing the notebook and updated record. A conflict leaves the newer source in place; the CLI preserves the submitted text for recovery.

The second check closes a race between loading and saving: the writer lock is released during transformation, so a different writer can save in that interval. The store reacquires its lock instead of nesting the same file lock. The notebook digest catches source changes, and the separate record revision catches CMS changes. If the transformed notebook is unchanged, `save()` returns without writing it.

Execution uses the same guarded-save boundary but a different transformation. `run()` loads a snapshot, executes an `nbclient` notebook with the source directory as its working directory, then copies outputs and execution counts into the original notebook structure. With `--index`, it executes a copy of the prefix but merges only the selected code cell's results. This avoids normalizing or overwriting the other cells' JSON. If the final save conflicts, Atlas preserves the executed notebook in `.atlas/cms/execution-conflicts/` for review.

The shared transformation is implemented in `src/atlas/notebook_ops.py`:

```python
def update(store: Store, name: str, expected_revision: str | None, transform) -> dict:
    with store.lock():
        loaded = load(store, name, expected_revision)
    notebook = copy.deepcopy(loaded["notebook"])
    transform(notebook)
    if len(json.dumps(notebook, ensure_ascii=False)) > 100_000_000:
        raise ValueError("Notebook is too large to save (>100 MB).")
    # write_notebook acquires the writer lock and checks both revisions. Holding
    # our read lock here would deadlock on its second file descriptor.
    return save(store, loaded, notebook)


def set_source(notebook: dict, index: int, source: str):
    if len(source) > MAX_SOURCE_CHARS:
        raise ValueError(f"Cell source exceeds {MAX_SOURCE_CHARS:,} characters.")
    if nb.source(notebook["cells"][index]) != source:
        notebook["cells"][index]["source"] = source.splitlines(keepends=True)
```

`transform` is a callable supplied by the command; for a source edit it calls `set_source`. Copying first lets the operation compare the result with the loaded snapshot. Only the source field is replaced, so a code cell's outputs remain until execution or clearing changes them.

`save()` then supplies two independent observations to the store:

```python
raw = nb.dump(notebook)
updated = store.write_notebook(
    loaded["record"]["id"],
    raw,
    loaded["metadata_revision"],
    digest(loaded["raw"]),
    images=images,
)
```

The record revision protects the CMS record; the digest protects the notebook bytes. The write runs under the store lock, but the checks are still necessary because the lock was released while the transform ran.

For execution, `run()` copies just the resulting fields back into the loaded notebook structure:

```python
updated = copy.deepcopy(loaded["notebook"])
for i in range(len(cells)) if index is None else [index]:
    if cells[i].cell_type == "code":
        updated["cells"][i]["outputs"] = target.cells[i].get("outputs", [])
        updated["cells"][i]["execution_count"] = target.cells[i].get(
            "execution_count"
        )
```

Here `target` is the executed notebook or prefix. The loop selects every cell for a full run, but only `index` for a targeted run. The prefix can establish kernel state without saving its new outputs into the original notebook.


## Run code on purpose

Notebook execution is separate from editing and rendering:

```sh
uv run at notebook kernels
uv run at notebook output post/understanding-attention --index 8
uv run at notebook run post/understanding-attention --index 8 --kernel python3 --timeout 300
uv run at notebook output post/understanding-attention --index 8
```

`output` reads saved output and extracts embedded images under `.atlas/cms/notebook-output/`. Running with `--index` executes the prefix in a fresh kernel, so earlier cells can establish state, but saves outputs only for the selected cell. Omitting `--index` runs the whole notebook. Code errors are saved and the command exits with status 1. An execution conflict preserves the results under `.atlas/cms/execution-conflicts/` rather than silently replacing a newer notebook.

If a result should no longer be kept, clear it deliberately:

```sh
uv run at notebook clear-outputs post/understanding-attention --index 8
```

`--from N` clears code outputs from that cell onward; omitting both a locator and `--from` clears all code-cell outputs. Those choices affect what a reader sees in a render, so I inspect before clearing.

## Review the source and the site

Before finishing, I re-read the changed cells and compare source against Git:

```sh
uv run at notebook cat post/understanding-attention --index 0 --with-revision --limit 0
uv run at notebook diff post/understanding-attention --base HEAD
uv run at cms check
uv run at cms sync --preview
```

`diff` focuses on cell source instead of embedded output JSON. It does not replace inspecting outputs when results changed. `check` includes publication blockers for drafts, so an unfinished draft can make it exit nonzero. Preview sync refreshes generated draft copies; `make preview` serves the draft-inclusive site, while `make build` renders published content. Both render without executing notebooks.

The result is a workflow in which the same `.ipynb` remains useful in a notebook editor and manageable from a terminal. The CMS owns the record, notebook commands operate on the content, and saved execution stays an explicit decision. The final article in this series looks at the [Vault](building-and-testing-a-vault-cli.qmd), Atlas's local secrets CLI and its audit history.



